# 21 | LangChain Middleware：敏感信息进入模型前，先脱敏

客服 Agent 有可能遇到用户发来的这种消息：

> 我的手机号是 13800138000，邮箱是 zhangsan@example.com，身份证号是 110101199003070013，帮我查一下退款进度。

Agent 可以处理退款，但模型不一定需要看到完整手机号、邮箱、身份证号。

`PIIMiddleware` 做的事很直接：**敏感信息进入模型前，先处理掉。**

本篇只看一件事：怎么让 Agent 还能工作，同时别把用户隐私原样丢给模型。

## 一、PII 和准备规则

`PII` 是可以识别到具体个人的信息，比如手机号、邮箱、身份证号、银行卡号、IP 地址。

LangChain 内置支持 `email`、`credit_card`、`ip`、`mac_address`、`url`。但是手机号、身份证号这类更偏地区和业务的信息，可以用自定义正则识别。

`PIIMiddleware` 不是合规保险箱，它更像模型入口前的一道安全门。门要装，钥匙也别乱放。

In [ ]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import AgentState, PIIDetectionError, PIIMiddleware, before_model
from langchain.chat_models import init_chat_model
from langchain.tools import tool

load_dotenv(dotenv_path=".env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
)

# 国内手机号：简单匹配 11 位手机号。
PHONE_PATTERN = r"(?<!\d)1[3-9]\d{9}(?!\d)"

# 身份证号：匹配 15 位或 18 位格式。这里用于演示，不等于完整实名校验。
ID_CARD_PATTERN = r"(?<!\d)(?:\d{17}[\dXx]|\d{15})(?!\d)"


def print_last_message(result):
    """只打印 Agent 最后一条回复，避免把完整消息列表刷屏。"""
    print(result["messages"][-1].content)


@before_model
def show_model_input(state: AgentState, runtime):
    """观察模型真正收到的最后一条用户消息。"""
    # 这个 Middleware 放在 PIIMiddleware 后面，用来查看脱敏后的输入。
    last_message = state["messages"][-1]
    print("[模型实际收到的用户消息]")
    print(last_message.content)
    return None

## 二、输入模型前先脱敏

用户消息里带了手机号、邮箱、身份证号。我们希望 Agent 能继续生成客服回复，但模型不要看到原始号码。

下面会打印“模型实际收到的用户消息”。重点看敏感信息是不是已经变成占位符。

In [ ]:
safe_agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        # 内置 email 检测：把邮箱替换成 [REDACTED_EMAIL]。
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        # 手机号不是内置类型，用自定义正则识别。
        PIIMiddleware(
            "phone_number",
            detector=PHONE_PATTERN,
            strategy="redact",
            apply_to_input=True,
        ),
        # 身份证号也用自定义正则识别。
        PIIMiddleware(
            "id_card",
            detector=ID_CARD_PATTERN,
            strategy="redact",
            apply_to_input=True,
        ),
        # 放在 PII 处理后面，用来观察脱敏后的模型输入。
        show_model_input,
    ],
    system_prompt="你是电商售后客服。回复要简洁，不要复述用户的完整隐私信息。",
)

result = safe_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "我的手机号是 13800138000，邮箱是 zhangsan@example.com，"
                    "身份证号是 110101199003070013。请帮我写一句客服回复，"
                    "告诉我会继续处理退款。"
                ),
            }
        ]
    }
)

print_last_message(result)

`apply_to_input=True` 表示：用户消息进入模型前先检查 PII。

如果输出里能看到类似下面的内容，就说明脱敏发生在模型调用前：

```text
我的手机号是 [REDACTED_PHONE_NUMBER]，邮箱是 [REDACTED_EMAIL]，身份证号是 [REDACTED_ID_CARD]。
```

脱敏不是让模型凭空判断业务结果。退款能不能继续，应该来自订单系统、身份认证系统或售后工具。模型需要的是 `order_status=未发货`、`identity_verified=True` 这类业务结果，不是完整手机号和身份证号。

## 三、四种策略怎么选

`PIIMiddleware` 检测到敏感信息后，可以用四种方式处理：

| 策略 | 作用 | 适合场景 |
|---|---|---|
| `redact` | 替换成占位符 | 通用脱敏 |
| `mask` | 保留少量可读字符 | 客服核对尾号 |
| `hash` | 替换成稳定 hash | 排查同一用户，但不看原文 |
| `block` | 直接阻止执行 | 高风险信息禁止进模型 |

先看 `hash` 和 `mask`。这段也会打印模型实际收到的内容。

In [ ]:
strategy_agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        # hash：同一个邮箱每次会得到同一个 hash，方便排查同一用户的问题。
        PIIMiddleware("email", strategy="hash", apply_to_input=True),
        # mask：信用卡只保留少量可读信息，适合人工核对尾号。
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        # 打印模型实际收到的内容，方便对比 hash 和 mask 的处理结果。
        show_model_input,
    ],
    system_prompt="你是客服助手。不要复述用户的完整邮箱或完整卡号。",
)

result = strategy_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "我的邮箱是 zhangsan@example.com，"
                    "信用卡号是 4111 1111 1111 1111。"
                    "请告诉我你已经收到信息。"
                ),
            }
        ]
    }
)

print_last_message(result)

`hash` 适合“知道是不是同一个用户，但不需要知道他是谁”。

`mask` 适合核对尾号，比如信用卡尾号。但它仍然保留了部分信息。如果字段完全不该进模型，就用 `redact` 或 `block`。

`block` 更狠一点：检测到就不调用模型。适合身份证号、银行卡号、内部密钥、访问令牌这类高风险信息。

In [ ]:
block_agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        # block：检测到身份证号就阻止继续执行。
        PIIMiddleware(
            "id_card",
            detector=ID_CARD_PATTERN,
            strategy="block",
            apply_to_input=True,
        )
    ],
)

try:
    block_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "我的身份证号是 110101199003070013，请帮我查退款。",
                }
            ]
        }
    )
except PIIDetectionError as e:
    print("检测到高风险 PII，已阻止进入模型。")
    print(type(e).__name__)

    # 真实业务里，拦截后应该给用户一个安全提示，而不是让请求静默失败。
    safe_reply = (
        "为了保护你的隐私，请不要在对话中发送完整身份证号。"
        "如果需要身份认证，请通过官方认证页面提交。"
    )
    print(safe_reply)

## 四、工具结果和输出也要保护

PII 不只来自用户输入，也可能来自工具结果。

比如 Agent 查客户资料，工具返回了邮箱和手机号。即使最终回复没泄露，模型也可能已经在下一轮调用里看到了原始联系方式。

这时要用：

- `apply_to_tool_results=True`：保护工具返回内容
- `apply_to_output=True`：保护模型最终输出

下面继续打印模型实际收到的消息。你可以注释掉这两个参数，对比工具结果是否会原样进入模型上下文。

In [ ]:
@tool
def lookup_customer_profile(order_id: str) -> str:
    """查询订单对应的客户联系信息。"""
    # 注意：真实项目里，工具内部日志也不要直接打印完整 PII。
    return "订单 O-1001 的客户邮箱是 zhangsan@example.com，手机号是 13800138000。"


tool_safe_agent = create_agent(
    model=model,
    tools=[lookup_customer_profile],
    middleware=[
        # 工具结果里出现邮箱时，也要在进入下一次模型调用前脱敏。
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_tool_results=True,
            apply_to_output=True,
        ),
        # 工具结果里的手机号同样处理。
        PIIMiddleware(
            "phone_number",
            detector=PHONE_PATTERN,
            strategy="redact",
            apply_to_tool_results=True,
            apply_to_output=True,
        ),
        # 观察工具结果进入下一次模型调用前，是否已经被脱敏。
        show_model_input,
    ],
    system_prompt=(
        "你是客服 Agent。用户要求查询客户资料时，必须调用 lookup_customer_profile。"
        "回复用户时，不要暴露完整联系方式，只说明已经找到联系方式并会继续跟进。"
    ),
)

result = tool_safe_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "请查询订单 O-1001 的客户资料，然后告诉我后续怎么联系。",
            }
        ]
    }
)

print_last_message(result)

这里要分清两件事：

- 最终回复没泄露，不代表模型没看到敏感信息。
- `PIIMiddleware` 保护的是 Agent 消息链路，不会自动清理你在工具函数、应用日志、追踪系统里手动记录的原始 PII。

生产里通常要同时做三层：Agent 链路脱敏、应用日志脱敏、观测平台脱敏。少一层，都可能在排查问题时给自己留惊喜。

## 五、小结

判断原则很简单：**模型只拿完成任务所需的最少信息。**

| 场景 | 推荐方式 |
|---|---|
| 普通脱敏 | `redact` |
| 核对尾号 | `mask` |
| 分析同一用户 | `hash` |
| 高风险信息禁止进入模型 | `block` |
| 工具返回了敏感信息 | `apply_to_tool_results=True` |
| 模型输出可能带敏感信息 | `apply_to_output=True` |

下一篇看成本和上下文窗口：对话越聊越长时，怎么让 Agent 先总结，再继续干活。

参考：

- LangChain Prebuilt Middleware：https://docs.langchain.com/oss/python/langchain/middleware/built-in
- LangChain PIIMiddleware API Reference：https://reference.langchain.com/python/langchain/agents/middleware/pii/PIIMiddleware